## 20. نقشه بازار

برای فروش آپارتمان مسکونی، حداقل موارد زیر بررسی شوند:

- میانه قیمت پیشنهادی هر مترمربع به تفکیک شهر
- میانه قیمت پیشنهادی هر مترمربع به تفکیک محله
- تعداد آگهی معتبر هر منطقه
- IQR یا شاخص پراکندگی
- درصد داده حذف‌شده در هر منطقه
- گران‌ترین و ارزان‌ترین محله قابل اعتماد

شهرهای مورد انتظار در تحلیل اصلی:

- تهران
- مشهد
- کرج
- اصفهان

### شرط رتبه‌بندی محله

یک محله صرفاً به‌دلیل داشتن میانه بالا یا پایین نباید رتبه‌بندی شود. تیم باید حداقل تعداد آگهی،
پوشش زمانی و کیفیت داده را کنترل کند.

ایمپورتها و خواندن فایل قبلی

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#1
df = pd.read_feather("../Outputs/19_df.feather")

In [ ]:
print("Shape:", df.shape)
print(df.columns.tolist())

ساخت دیتای فروش آپارتمان مسکونی

In [ ]:
#2
sell_df = df[
    df["cat2_slug"] == "residential-sell"
].copy()

print("Residential sale listings:", len(sell_df))

محاسبه قیمت پیشنهادی هر متر مربع

In [ ]:
#3
sell_df["price_per_sqm"] = (
        sell_df["price_value"] /
        sell_df["building_size"]
    )

print(
    sell_df["price_per_sqm"]
    .describe()
)

ساخت دیتای اولیه برای تحلیل(اینحا فقط قیمت و مساحت مثبت را وارد محاسبات میکنیم)

In [ ]:
#4
analysis_df = sell_df[
    (sell_df["price_value"] > 0) &
    (sell_df["building_size"] > 0) &
    (sell_df["price_per_sqm"] > 0)
].copy()

print("Initial valid records:", len(analysis_df))

تعیین پرت ها

In [ ]:
#5
LOWER_PRICE_LIMIT = 1_000_000

p99 = analysis_df["price_per_sqm"].quantile(0.99)

analysis_df["invalid_ppsqm_low_flag"] = (
    analysis_df["price_per_sqm"] < LOWER_PRICE_LIMIT
)

analysis_df["extreme_ppsqm_high_flag"] = (
    analysis_df["price_per_sqm"] > p99
)

print("P99:", p99)

print(
    "Low price outliers:",
    analysis_df["invalid_ppsqm_low_flag"].sum()
)

print(
    "High price outliers:",
    analysis_df["extreme_ppsqm_high_flag"].sum()
)

ساخت دیتافریم جدید با داده های قابل اتکا

In [ ]:
#6
analysis_valid = analysis_df[
    (analysis_df["price_per_sqm"] >= LOWER_PRICE_LIMIT) &
    (analysis_df["price_per_sqm"] <= p99)
].copy()

print("Analysis-valid records:", len(analysis_valid))

ساخت جدول محله ها
/ از همه آگهی ها فروش استفاده میکنیم اما قیمت هارا فقط از analysis_valid میگیریم

In [ ]:
#7
neighborhood_base = (
    sell_df[
        sell_df["city_slug"].notna() &
        sell_df["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        total_listing_count=("price_value", "size"),
        missing_price_count=("price_value", lambda x: x.isna().sum())
    )
    .reset_index()
)

neighborhood_base["missing_price_rate"] = (
    neighborhood_base["missing_price_count"] /
    neighborhood_base["total_listing_count"]
)

neighborhood_base.head()

آمار قیمت برای هر محله

In [ ]:
#8
neighborhood_price_summary = (
    analysis_valid[
        analysis_valid["city_slug"].notna() &
        analysis_valid["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["price_per_sqm"]
    .agg(
        valid_listing_count="count",
        median_price_per_sqm="median",
        p25_price_per_sqm=lambda x: x.quantile(0.25),
        p75_price_per_sqm=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

neighborhood_price_summary.head()

محاسیه IQR

In [ ]:
#9
neighborhood_price_summary["iqr_price_per_sqm"] = (
    neighborhood_price_summary["p75_price_per_sqm"] -
    neighborhood_price_summary["p25_price_per_sqm"]
)

نرخ اوتلایر برای هر محله

In [ ]:
#10
neighborhood_outliers = (
    analysis_df[
        analysis_df["city_slug"].notna() &
        analysis_df["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        price_valid_before_outlier_filter=("price_per_sqm", "count"),
        outlier_count=(
            "price_per_sqm",
            lambda x: (
                (x < LOWER_PRICE_LIMIT) |
                (x > p99)
            ).sum()
        )
    )
    .reset_index()
)

neighborhood_outliers["outlier_rate"] = (
    neighborhood_outliers["outlier_count"] /
    neighborhood_outliers["price_valid_before_outlier_filter"]
)

neighborhood_outliers.head()

بررسی پوشش زمانی

In [ ]:
#11
sell_df["created_at_month"] = pd.to_datetime(
    sell_df["created_at_month"],
    errors="coerce"
)

analysis_valid["created_at_month"] = pd.to_datetime(
    analysis_valid["created_at_month"],
    errors="coerce"
)

total_months = (
    sell_df["created_at_month"]
    .dropna()
    .nunique()
)

print("Total months in dataset:", total_months)

برای هر محله تعداد ماه هایی که واقعا آگهی معتبر داشته اند را محاسبه میکنیم.

In [ ]:
neighborhood_time = (
    analysis_valid[
        analysis_valid["city_slug"].notna() &
        analysis_valid["neighborhood_slug"].notna() &
        analysis_valid["created_at_month"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["created_at_month"]
    .nunique()
    .rename("month_count")
    .reset_index()
)

neighborhood_time["time_coverage_rate"] = (
    neighborhood_time["month_count"] /
    total_months
)

neighborhood_time.head()

ساخت neighborhood_market_summary/ تمام جدول ها را به هم وصل میکنیم

In [ ]:
#12
neighborhood_market_summary = (
    neighborhood_base
    .merge(
        neighborhood_price_summary,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_outliers[
            [
                "city_slug",
                "neighborhood_slug",
                "outlier_rate"
            ]
        ],
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_time,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
)

neighborhood_market_summary.head()

پر کردن مواردی واقعا صفر هستند/ اینجا صفر منطقی است

In [ ]:
neighborhood_market_summary["valid_listing_count"] = (
    neighborhood_market_summary["valid_listing_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["month_count"] = (
    neighborhood_market_summary["month_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["time_coverage_rate"] = (
    neighborhood_market_summary["time_coverage_rate"]
    .fillna(0)
)

neighborhood_market_summary["outlier_rate"] = (
    neighborhood_market_summary["outlier_rate"]
    .fillna(0)
)

تعریف Reliability

حداقل 50 آگهی معتبر/ حداقل 3 ماه پوشش زمانی/ حداقل 50درصد پوشش زمانی/ نرخ داده میسینگ حداکثر 30 درصد/ نرخ پرت حداکثر 10درصد

In [ ]:
#14
MIN_VALID_LISTINGS = 50
MIN_MONTHS = 3
MIN_TIME_COVERAGE = 0.50
MAX_MISSING_PRICE_RATE = 0.30
MAX_OUTLIER_RATE = 0.10

neighborhood_market_summary["reliability_flag"] = np.where(
    (
        (neighborhood_market_summary["valid_listing_count"] >= MIN_VALID_LISTINGS) &
        (neighborhood_market_summary["month_count"] >= MIN_MONTHS) &
        (neighborhood_market_summary["time_coverage_rate"] >= MIN_TIME_COVERAGE) &
        (neighborhood_market_summary["missing_price_rate"] <= MAX_MISSING_PRICE_RATE) &
        (neighborhood_market_summary["outlier_rate"] <= MAX_OUTLIER_RATE)
    ),
    "Reliable",
    "Low_reliability"
)

فقط شهرهای اصلی پروژه

In [ ]:
#15
target_cities = [
    "tehran",
    "mashhad",
    "karaj",
    "isfahan"
]

neighborhood_market_summary = (
    neighborhood_market_summary[
        neighborhood_market_summary["city_slug"].isin(target_cities)
    ]
    .copy()
)

print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

مرتب سازی نهایی ستون ها

In [ ]:
#16
neighborhood_market_summary = neighborhood_market_summary[
    [
        "city_slug",
        "neighborhood_slug",
        "total_listing_count",
        "valid_listing_count",
        "median_price_per_sqm",
        "p25_price_per_sqm",
        "p75_price_per_sqm",
        "iqr_price_per_sqm",
        "missing_price_rate",
        "outlier_rate",
        "month_count",
        "time_coverage_rate",
        "reliability_flag"
    ]
].sort_values(
    ["city_slug", "median_price_per_sqm"],
    ascending=[True, False]
)

neighborhood_market_summary.head(20)

بررسی کیفیت جدول

In [ ]:
#17
print("Shape:")
print(neighborhood_market_summary.shape)

print("\nReliability:")
print(
    neighborhood_market_summary["reliability_flag"]
    .value_counts()
)

print("\nMissing values:")
print(
    neighborhood_market_summary.isna().sum()
)

print("\nTarget cities:")
print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

اصلاح پوشش زمانی

In [ ]:
#17.5 
# The dataset contains a small number of records before May 2024
# and after December 2024.
# From May 2024 onward, listing volume becomes substantially larger
# and more stable.
#
# Therefore, for neighborhood-level market mapping and reliability
# assessment, we use a six-month high-coverage window:
# May 2024 to October 2024.

project_months = pd.period_range(
    start="2024-05",
    end="2024-10",
    freq="M"
)

print("Selected analysis months:")
print(project_months)

In [ ]:
#17.6
sell_project = sell_df[
    sell_df["created_at_month"]
    .dt.to_period("M")
    .isin(project_months)
].copy()

analysis_valid_project = analysis_valid[
    analysis_valid["created_at_month"]
    .dt.to_period("M")
    .isin(project_months)
].copy()

print("Total residential-sale listings in selected period:")
print(len(sell_project))

print("\nValid price listings in selected period:")
print(len(analysis_valid_project))

print("\nMonthly listing counts:")
print(
    sell_project["created_at_month"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

In [ ]:
#17.7
total_months = len(project_months)

neighborhood_time = (
    analysis_valid_project[
        analysis_valid_project["city_slug"].notna() &
        analysis_valid_project["neighborhood_slug"].notna() &
        analysis_valid_project["created_at_month"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["created_at_month"]
    .nunique()
    .rename("month_count")
    .reset_index()
)

neighborhood_time["time_coverage_rate"] = (
    neighborhood_time["month_count"] /
    total_months
)

neighborhood_time.head()

ساخت دوباره بیس با تایم پوشش زمانی اصلاح شده

In [ ]:
#18
neighborhood_base = (
    sell_project[
        sell_project["city_slug"].notna() &
        sell_project["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        total_listing_count=("price_value", "size"),
        missing_price_count=(
            "price_value",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

neighborhood_base["missing_price_rate"] = (
    neighborhood_base["missing_price_count"] /
    neighborhood_base["total_listing_count"]
)

آمار قیمت

In [ ]:
#19
neighborhood_price_summary = (
    analysis_valid_project[
        analysis_valid_project["city_slug"].notna() &
        analysis_valid_project["neighborhood_slug"].notna()
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )["price_per_sqm"]
    .agg(
        valid_listing_count="count",
        median_price_per_sqm="median",
        p25_price_per_sqm=lambda x: x.quantile(0.25),
        p75_price_per_sqm=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

neighborhood_price_summary["iqr_price_per_sqm"] = (
    neighborhood_price_summary["p75_price_per_sqm"] -
    neighborhood_price_summary["p25_price_per_sqm"]
)

نرخ اوتلایر برحسب محله

In [ ]:
#20
neighborhood_outliers = (
    analysis_df[
        analysis_df["city_slug"].notna() &
        analysis_df["neighborhood_slug"].notna() &
        analysis_df["created_at_month"].dt.to_period("M").isin(project_months)
    ]
    .groupby(
        ["city_slug", "neighborhood_slug"],
        observed=True
    )
    .agg(
        price_valid_before_outlier_filter=(
            "price_per_sqm",
            "count"
        ),
        outlier_count=(
            "price_per_sqm",
            lambda x: (
                (x < LOWER_PRICE_LIMIT) |
                (x > p99)
            ).sum()
        )
    )
    .reset_index()
)

neighborhood_outliers["outlier_rate"] = (
    neighborhood_outliers["outlier_count"] /
    neighborhood_outliers["price_valid_before_outlier_filter"]
)

ساخت جدول نهایی

In [ ]:
#21
neighborhood_market_summary = (
    neighborhood_base
    .merge(
        neighborhood_price_summary,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_outliers[
            [
                "city_slug",
                "neighborhood_slug",
                "outlier_rate"
            ]
        ],
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
    .merge(
        neighborhood_time,
        on=["city_slug", "neighborhood_slug"],
        how="left"
    )
)

neighborhood_market_summary["valid_listing_count"] = (
    neighborhood_market_summary["valid_listing_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["month_count"] = (
    neighborhood_market_summary["month_count"]
    .fillna(0)
    .astype(int)
)

neighborhood_market_summary["time_coverage_rate"] = (
    neighborhood_market_summary["time_coverage_rate"]
    .fillna(0)
)

neighborhood_market_summary["outlier_rate"] = (
    neighborhood_market_summary["outlier_rate"]
    .fillna(0)
)

Reliability نهایی

In [ ]:
#22
MIN_VALID_LISTINGS = 50
MIN_MONTHS = 3
MIN_TIME_COVERAGE = 0.50
MAX_MISSING_PRICE_RATE = 0.30
MAX_OUTLIER_RATE = 0.10

neighborhood_market_summary["reliability_flag"] = np.where(
    (
        (neighborhood_market_summary["valid_listing_count"] >= MIN_VALID_LISTINGS) &
        (neighborhood_market_summary["month_count"] >= MIN_MONTHS) &
        (neighborhood_market_summary["time_coverage_rate"] >= MIN_TIME_COVERAGE) &
        (neighborhood_market_summary["missing_price_rate"] <= MAX_MISSING_PRICE_RATE) &
        (neighborhood_market_summary["outlier_rate"] <= MAX_OUTLIER_RATE)
    ),
    "Reliable",
    "Low_reliability"
)

فقط 4 شهر اصلی

In [ ]:
#23
target_cities = [
    "tehran",
    "mashhad",
    "karaj",
    "isfahan"
]

neighborhood_market_summary = (
    neighborhood_market_summary[
        neighborhood_market_summary["city_slug"].isin(target_cities)
    ]
    .copy()
)

ترتیب ستون ها

In [ ]:
#24
neighborhood_market_summary = neighborhood_market_summary[
    [
        "city_slug",
        "neighborhood_slug",
        "total_listing_count",
        "valid_listing_count",
        "median_price_per_sqm",
        "p25_price_per_sqm",
        "p75_price_per_sqm",
        "iqr_price_per_sqm",
        "missing_price_rate",
        "outlier_rate",
        "month_count",
        "time_coverage_rate",
        "reliability_flag"
    ]
].sort_values(
    ["city_slug", "median_price_per_sqm"],
    ascending=[True, False]
)

چک نهایی

In [ ]:
#25
print("Shape:")
print(neighborhood_market_summary.shape)

print("\nReliability:")
print(
    neighborhood_market_summary["reliability_flag"]
    .value_counts()
)

print("\nMissing values:")
print(
    neighborhood_market_summary.isna().sum()
)

print("\nTarget cities:")
print(
    neighborhood_market_summary["city_slug"]
    .value_counts()
)

بررسی منطقی Reliability

In [ ]:
#26
reliability_check = neighborhood_market_summary.copy()

reliability_check["check_listing"] = (
    reliability_check["valid_listing_count"] >= MIN_VALID_LISTINGS
)

reliability_check["check_month"] = (
    reliability_check["month_count"] >= MIN_MONTHS
)

reliability_check["check_coverage"] = (
    reliability_check["time_coverage_rate"] >= MIN_TIME_COVERAGE
)

reliability_check["check_missing"] = (
    reliability_check["missing_price_rate"] <= MAX_MISSING_PRICE_RATE
)

reliability_check["check_outlier"] = (
    reliability_check["outlier_rate"] <= MAX_OUTLIER_RATE
)

print(
    reliability_check[
        [
            "check_listing",
            "check_month",
            "check_coverage",
            "check_missing",
            "check_outlier"
        ]
    ].apply(pd.Series.value_counts)
)

Reliable neighborhoods

In [ ]:
#27
reliable_neighborhoods = (
    neighborhood_market_summary[
        neighborhood_market_summary["reliability_flag"] == "Reliable"
    ]
    .copy()
)

print(
    "Reliable neighborhoods:",
    len(reliable_neighborhoods)
)

گرانترین و ارزانترین محله ها

In [ ]:
#28
for city in target_cities:

    city_data = (
        reliable_neighborhoods[
            reliable_neighborhoods["city_slug"] == city
        ]
        .sort_values(
            "median_price_per_sqm",
            ascending=False
        )
    )

    print("\n" + "=" * 70)
    print(city.upper())
    print("=" * 70)

    if city_data.empty:
        print("No reliable neighborhood found.")
        continue

    print("\nMost expensive reliable neighborhood:")
    print(
        city_data[
            [
                "neighborhood_slug",
                "valid_listing_count",
                "median_price_per_sqm",
                "iqr_price_per_sqm",
                "missing_price_rate",
                "outlier_rate",
                "month_count",
                "time_coverage_rate"
            ]
        ].head(1)
    )

    print("\nCheapest reliable neighborhood:")
    print(
        city_data[
            [
                "neighborhood_slug",
                "valid_listing_count",
                "median_price_per_sqm",
                "iqr_price_per_sqm",
                "missing_price_rate",
                "outlier_rate",
                "month_count",
                "time_coverage_rate"
            ]
        ].tail(1)
    )

10 محاه گران هر شهر

In [ ]:
#29
top_expensive_neighborhoods = (
    reliable_neighborhoods
    .sort_values(
        ["city_slug", "median_price_per_sqm"],
        ascending=[True, False]
    )
    .groupby(
        "city_slug",
        observed=True
    )
    .head(10)
)

top_expensive_neighborhoods[
    [
        "city_slug",
        "neighborhood_slug",
        "valid_listing_count",
        "median_price_per_sqm",
        "iqr_price_per_sqm",
        "time_coverage_rate"
    ]
]

10 محله ارزان هر شهر

In [ ]:
#30
top_cheap_neighborhoods = (
    reliable_neighborhoods
    .sort_values(
        ["city_slug", "median_price_per_sqm"],
        ascending=[True, True]
    )
    .groupby(
        "city_slug",
        observed=True
    )
    .head(10)
)

top_cheap_neighborhoods[
    [
        "city_slug",
        "neighborhood_slug",
        "valid_listing_count",
        "median_price_per_sqm",
        "iqr_price_per_sqm",
        "time_coverage_rate"
    ]
]

سیو خروجی

In [ ]:
#31
neighborhood_market_summary.to_feather(
    "../Outputs/market_map.feather"
)

print("Final market map saved successfully.")

خلاصه نهایی

In [ ]:
#32
print(
    neighborhood_market_summary[
        [
            "city_slug",
            "neighborhood_slug",
            "valid_listing_count",
            "median_price_per_sqm",
            "iqr_price_per_sqm",
            "missing_price_rate",
            "outlier_rate",
            "month_count",
            "time_coverage_rate",
            "reliability_flag"
        ]
    ].head(20)
)

In [ ]:
neighborhood_market_summary.columns

سیو فایل

In [ ]:
neighborhood_market_summary.to_csv("../Outputs/market_map.csv")

In [ ]:
# Code

# محل پیاده‌سازی تیم:
# یک جدول neighborhood_market_summary بسازید.
#
# ستون‌های پیشنهادی:
# city
# neighborhood
# valid_listing_count
# median_price_per_sqm
# p25_price_per_sqm
# p75_price_per_sqm
# missing_price_rate
# outlier_rate
# reliability_flag